In [1]:
# Import packages
import os
import sys
import mlflow
sys.path.append(r"C:\Users\Admin\WORK\Project_CV\Model_NLP_sentiment\src")
from dataset import trainset, testset, word2int
import torch
from torch import nn 
from torch import optim
from torch.utils.data import DataLoader
from tqdm import tqdm    
from LSTM_model import RNN




100%|██████████| 100/100 [00:00<?, ?it/s]


In [ ]:
'''Before begin to start model should do some steps:
1. In <preprocessing.py> Choose  the types of data preparation for NLP: True or False
2. In <dataset.py>  Choose count of review  for test model (example df.sample(100))  or for work with whole dataset delete  df.sample(100)
'''

In [3]:
# create dataloaders
trainloader = DataLoader(trainset, shuffle=True, batch_size=80)
testloader = DataLoader(testset, shuffle=True, batch_size=80)

In [4]:
print(f"Dataset size: {len(trainset)}")
print(f"Number of batches: {len(trainloader)}")
print(f"Batch size: {trainloader.batch_size}")

# Size of one batch
sample_batch = next(iter(trainloader))
if isinstance(sample_batch, (list, tuple)):
    print(f"Sample batch token shape: {sample_batch[0].shape}")
    print(f"Sample batch labels shape: {sample_batch[1].shape}")
    
gg,kk =sample_batch
print(gg[0],kk[0])

Dataset size: 80
Number of batches: 1
Batch size: 80
Sample batch token shape: torch.Size([80, 256])
Sample batch labels shape: torch.Size([80])
tensor([ 678, 1536,  504, 1356,  262, 1537, 1256, 3163, 1538,   45,  906, 1287,
        3164, 1141, 3165, 3166, 3167,  105,  815, 3168, 1539, 1540,  242,  182,
        1541,   56, 3169,   42,  406,   38,  883,   72, 3170, 3171,   35,  678,
        3172,  555,    5,  262,   53,  170,  904, 1343,  891, 3173, 3174,  165,
        3175,  232,  264, 1063, 1537,  207, 3176, 1279,    4,  619,   53, 1169,
         736, 1542,  149, 3177, 3178, 1440, 3179, 3180, 1216, 3181,  170, 3182,
        3183, 1540, 1516, 3184, 3185, 3186, 1304,   24, 3187,  287,  226,  679,
         286,   14,   60,  343, 1543, 3188,  443, 3189,  443,  170,  241,  549,
        3190,  145,  188, 3191, 3192,  262, 3193,    5,   26, 3194, 1538,  678,
        1536, 1295,  236,  355, 1544, 3195,  149, 3196,  364, 1418,  484, 1329,
         803,  804,    0,    0,    0,    0,    0,    0,

In [ ]:
#For starting MLflow server change the path to folders, copy next commit  and run to cmd:


# mlflow server --backend-store-uri "file:///C:Users/Admin/ML_flow_Tracking/data_local" --default-artifact-root "file:///C:Users/Admin/ML_flow_Tracking/artefacts" --host localhost --port 5000

In [5]:
# Indicate username for registration at the MLflow server
os.environ['USER'] = 'Evgenii_K'

In [6]:
# for reproducibility of results
seed = 42
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

In [7]:
# Initialization of  MLflow
# import mlflow.experiments
mlflow.set_tracking_uri('http://127.0.0.1:5000/')

# Indicate project name
mlflow.set_experiment('Sentiment_analysis')

<Experiment: artifact_location='file:///C:Users/Admin/WORK/Project_CV/Model_NLP_sentiment/ML_flow_Tracking/artefacts/277569375052305817', creation_time=1754376415356, experiment_id='277569375052305817', last_update_time=1754376415356, lifecycle_stage='active', name='Sentiment_analysis', tags={}>

In [8]:
# Set off MLflow warnings 
import logging
mlflow_logger = logging.getLogger("mlflow")
mlflow_logger.setLevel(logging.ERROR)

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

#Set hyperparameter
EPOCH = 10
LR = 0.001
momentum = 0
weight_decay = 0
opt = 'Adam' 
run_name='Exp_1'
vocab_size = len(word2int)
output_size = 1
embedding_size = 256
hidden_size = 512
n_layers = 2
dropout=0.25



NameError: name 'torch' is not defined

In [10]:
model = RNN(vocab_size, output_size, hidden_size, embedding_size, n_layers, dropout)
print(model)

RNN(
  (embedding): Embedding(4027, 512)
  (lstm): LSTM(512, 256, num_layers=2, batch_first=True, dropout=0.25)
  (fc1): Linear(in_features=256, out_features=256, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=256, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


In [17]:
# train loop
model = model.to(device)

epochloop = tqdm(range(epochs), position=0, desc='Training', leave=True)

# early stop trigger
es_trigger = 0
val_loss_min = torch.inf

for e in epochloop:

    #################
    # training mode #
    #################

    model.train()

    train_loss = 0
    train_acc = 0

    for id, (feature, target) in enumerate(trainloader):
        # add epoch meta info
        epochloop.set_postfix_str(f'Training batch {id}/{len(trainloader)}')

        # move to device
        feature, target = feature.to(device), target.to(device)

        # reset optimizer
        optim.zero_grad()

        # forward pass
        out = model(feature)

        # acc
        predicted = torch.tensor([1 if i == True else 0 for i in out > 0.5], device=device)
        equals = predicted == target
        acc = torch.mean(equals.type(torch.FloatTensor))
        train_acc += acc.item()

        # loss
        loss = criterion(out.squeeze(), target.float())
        train_loss += loss.item()
        loss.backward()

        # clip grad
        nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

        # update optimizer
        optim.step()

        # free some memory
        del feature, target, predicted

    history['train_loss'].append(train_loss / len(trainloader))
    history['train_acc'].append(train_acc / len(trainloader))

    ####################
    # validation model #
    ####################

    model.eval()

    val_loss = 0
    val_acc = 0

    with torch.no_grad():
        for id, (feature, target) in enumerate(valloader):
            # add epoch meta info
            epochloop.set_postfix_str(f'Validation batch {id}/{len(valloader)}')
            
            # move to device
            feature, target = feature.to(device), target.to(device)

            # forward pass
            out = model(feature)

            # acc
            predicted = torch.tensor([1 if i == True else 0 for i in out > 0.5], device=device)
            equals = predicted == target
            acc = torch.mean(equals.type(torch.FloatTensor))
            val_acc += acc.item()

            # loss
            loss = criterion(out.squeeze(), target.float())
            val_loss += loss.item()

            # free some memory
            del feature, target, predicted

        history['val_loss'].append(val_loss / len(valloader))
        history['val_acc'].append(val_acc / len(valloader))

    # reset model mode
    model.train()

    # add epoch meta info
    epochloop.set_postfix_str(f'Val Loss: {val_loss / len(valloader):.3f} | Val Acc: {val_acc / len(valloader):.3f}')

    # print epoch
    if (e+1) % print_every == 0:
        epochloop.write(f'Epoch {e+1}/{epochs} | Train Loss: {train_loss / len(trainloader):.3f} Train Acc: {train_acc / len(trainloader):.3f} | Val Loss: {val_loss / len(valloader):.3f} Val Acc: {val_acc / len(valloader):.3f}')
        epochloop.update()

    # save model if validation loss decrease
    if val_loss / len(valloader) <= val_loss_min:
        torch.save(model.state_dict(), './sentiment_lstm.pt')
        val_loss_min = val_loss / len(valloader)
        es_trigger = 0
    else:
        epochloop.write(f'[WARNING] Validation loss did not improved ({val_loss_min:.3f} --> {val_loss / len(valloader):.3f})')
        es_trigger += 1

    # force early stop
    if es_trigger >= es_limit:
        epochloop.write(f'Early stopped at Epoch-{e+1}')
        # update epochs history
        history['epochs'] = e+1
        break

Training:   0%|          | 0/8 [00:00<?, ?it/s, Training batch 0/1]

: 

In [11]:
# Starting MLflow 

with mlflow.start_run(run_name = run_name) as run:                            # starting experiment with name "run_name"
    net = model.to(device)
    loss_func = nn.BCELoss()                                                 # Binary Cross-Entropy Loss
    optimizer = optim.Adam(net.parameters(), lr=LR, 
                          weight_decay = weight_decay)                         #momentum=0.9, weight_decay=5e-4 
    
    mlflow.log_param("momentum", momentum)
    mlflow.log_param("weight_decay", weight_decay)
    mlflow.log_param("lr", LR)
    mlflow.log_param("optimizer", opt)
    mlflow.log_param("epochs", EPOCH )
    
    maxacc = 0
    itr_record = 0

    for epoch in range(EPOCH):
        epoch += 1
        net.train()
        train_loss = 0.0
        test_loss = 0.0
        max_train_acc = 0
        max_test_acc = 0
        correct = 0.0
        train_samples = 0.0
        test_samples = 0.0

        print(f'Началось обучение {epoch} эпохи')
        
        for itr, data in enumerate(trainloader):
            if itr == 5:                                                      # to stop the model for checking
                break
            inputs, labels = data 
            inputs, labels = inputs.to(device), labels.to(device).float() 
           
            outputs = net(inputs)
            loss = loss_func(outputs, labels.unsqueeze(1))                    #  Формат [batch, 1]
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * inputs.size(0)
        
            # train_loss += loss.item() * outputs.size(0)                          # train_loss+= mean_batch_loss * batch_size 
                                                                                 #  Multiplication by outputs.size(0) (batch) is  a transformation of the average
                                                                                 #  LOSS value for the batch into the total.  
            predicted = (outputs > 0.5).float()
            correct += (predicted == labels.unsqueeze(1)).float().sum()
            train_samples += inputs.size(0)                            # _,#  predicted - value tensor, number of index with max value.
                                                                                 # .data #  link two tensors        
            # train_samples += outputs.size(0)                                     #  Counts the number of photos.
            # correct += predicted.eq(labels.data).cpu().sum()                     # Sums up the number of matching  with labels.    
                        
        train_loss /= train_samples
        train_acc = 100*correct / train_samples                                  # Accuracy  
        
        mlflow.log_metric("train_loss", train_loss, step=epoch)
        mlflow.log_metric("train_acc", train_acc, step=epoch)
        print(f'The Epoch {epoch}:')
        print(f'Train loss - {train_loss:.3f}, Train accuracy - {train_acc:.2f} %')

        net.eval()
        
        
        correct = 0
        
        with torch.no_grad():
            for itr, data in enumerate(testloader):
                if itr == 5:
                    break
                inputs, labels = data 
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = net(inputs)
                loss = loss_func(outputs, labels)

                test_loss += loss.item() * outputs.size(0)
                _, predicted = torch.max(outputs.data, 1)                  
                test_samples += outputs.size(0)                            
            
                correct += predicted.eq(labels.data).cpu().sum()

        test_loss /= test_samples
        test_acc = 100*correct / test_samples
       
        mlflow.log_metric("test_loss", test_loss, step=epoch)
        mlflow.log_metric("test_acc", test_acc, step=epoch)
        print(f'Test loss - {test_loss:.3f}, Test accuracy - {test_acc:.2f} %')

        if test_acc > maxacc:
            print('Saving model because its better')
            maxacc = test_acc
            mlflow.pytorch.log_model(net, "MODEL_NAME")                                          # Indicate model name "MODEL_NAME"
        print('-------')

    print(f'Max accuracy - {maxacc:.2f} %')
    mlflow.log_metric("max test accuracy", maxacc)

mlflow.end_run()

Началось обучение 1 эпохи


: 

In [6]:
# define training device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cpu


In [8]:
# training config
lr = 0.001
criterion = nn.BCELoss()  # we use BCELoss cz we have binary classification problem
optim = torch.optim.Adam(model.parameters(), lr=lr)
grad_clip = 5
epochs = 8
print_every = 1
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
    'epochs': epochs
}
es_limit = 5